# Benchmark pipeline - corpus Vaucluse

Ce notebook sert a verifier l'annotation manuelle, calculer les metriques du benchmark et produire des graphiques presentables.

Fichiers utilises :

- `corpus_benchmark_annotation.json` : fichier d'annotation manuelle
- `metadata_output_benchmark_300.json` : copie filtree du JSON source benchmark
- `corpus_benchmark/` et `corpus_benchmark2/` : images du benchmark
- `guide_benchmark.md` : regles d'annotation

Workflow recommande :

1. lancer les cellules de preparation
2. utiliser la fonction `review_card(...)` pour controler des cartes si besoin
3. completer `corpus_benchmark_annotation.json` en suivant le guide
4. relancer les cellules de validation, metriques et graphiques


In [1]:
import json
import math
import unicodedata
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, Markdown, display

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 50)

COLORS = {
    "correct": "#1f9d72",
    "error": "#d1495b",
    "missing": "#9aa5b1",
    "coverage": "#2b6cb0",
    "accuracy": "#2f855a",
    "accepted": "#dd6b20",
    "cross": "#6b46c1",
    "bonus": "#d69e2e",
    "excluded": "#4a5568",
    "hallucination": "#b83280",
}

FIELD_CONFIG = {
    "city": {
        "label": "Commune",
        "pred_col": "trans_city",
        "error_col": "error_city",
        "special_value": "Aucune ville",
    },
    "hamlet": {
        "label": "Lieu-dit",
        "pred_col": "trans_hamlet_uniformise",
        "error_col": "error_hamlet",
        "special_value": "Aucun lieu-dit",
    },
    "monument": {
        "label": "Monument",
        "pred_col": "trans_monument_uniformise",
        "error_col": "error_monument",
        "special_value": "Aucun monument",
    },
}

ALLOWED_ERRORS = {
    "error_city": {
        "mauvaise_commune",
        "commune_absente",
        "commune_hallucinee",
        "commune_lieu_edition",
        "commune_mauvais_niveau",
        "commune_normalisation",
    },
    "error_hamlet": {
        "mauvais_lieudit",
        "lieudit_absent",
        "lieudit_hallucine",
        "lieudit_mauvais_niveau",
        "lieudit_normalisation",
    },
    "error_monument": {
        "mauvais_monument",
        "monument_absent",
        "monument_hallucine",
        "monument_partiel",
        "monument_mauvais_niveau",
        "monument_multiple_incomplet",
        "monument_multiple_excedentaire",
        "monument_normalisation",
    },
    "another_error": {
        "swap_commune_lieudit",
        "swap_commune_monument",
        "swap_lieudit_monument",
        "decalage_general_niveaux",
        "source_json_absente",
        "cas_ambigu",
    },
}

HALLUCINATION_CODES = {
    "Commune": "commune_hallucinee",
    "Lieu-dit": "lieudit_hallucine",
    "Monument": "monument_hallucine",
}

ALLOWED_BONUS = {"find_else"}

EXPECTED_COLUMNS = [
    "File name",
    "trans_city",
    "trans_hamlet_uniformise",
    "trans_monument_uniformise",
    "error_city",
    "error_hamlet",
    "error_monument",
    "another_error",
    "bonus",
]

def find_root():
    candidates = [Path.cwd(), Path.cwd() / "benchmark_300_cartes_vaucluse"]
    for candidate in candidates:
        if (candidate / "corpus_benchmark_annotation.json").exists():
            return candidate
    raise FileNotFoundError("Impossible de trouver le dossier benchmark_300_cartes_vaucluse")

ROOT = find_root()
ANNOTATION_PATH = ROOT / "corpus_benchmark_annotation.json"
FILTERED_JSON_PATH = ROOT / "metadata_output_benchmark_300.json"
GUIDE_PATH = ROOT / "guide_benchmark.md"
CORPUS_DIRS = sorted([p for p in ROOT.iterdir() if p.is_dir() and p.name.startswith("corpus_benchmark")])
PLOTS_DIR = ROOT / "plots_benchmark_300"
EXPORT_DIR = ROOT / "exports_benchmark_300"
PLOTS_DIR.mkdir(exist_ok=True)
EXPORT_DIR.mkdir(exist_ok=True)

IMAGE_PATH_BY_NAME = {}
DUPLICATE_IMAGE_NAMES = set()
for corpus_dir in CORPUS_DIRS:
    for image_path in corpus_dir.glob("*.jpg"):
        if image_path.name in IMAGE_PATH_BY_NAME:
            DUPLICATE_IMAGE_NAMES.add(image_path.name)
        else:
            IMAGE_PATH_BY_NAME[image_path.name] = image_path
EXPECTED_IMAGE_COUNT = len(IMAGE_PATH_BY_NAME)

def is_blank(value):
    if value is None:
        return True
    if isinstance(value, float) and math.isnan(value):
        return True
    text = str(value).strip()
    return text == "" or text.lower() in {"none", "null", "nan"}

def normalize_text(value):
    if is_blank(value):
        return ""
    text = unicodedata.normalize("NFKD", str(value))
    text = "".join(ch for ch in text if not unicodedata.combining(ch))
    return " ".join(text.lower().strip().split())

def parse_multi_codes(value):
    if is_blank(value):
        return []
    return [token.strip() for token in str(value).split(";") if token.strip()]

print(f"ROOT: {ROOT}")
print(f"Annotation: {ANNOTATION_PATH.name}")
print(f"Corpus dirs: {[p.name for p in CORPUS_DIRS]}")
print(f"Images indexees: {EXPECTED_IMAGE_COUNT}")
print(f"Exports: {EXPORT_DIR}")
if DUPLICATE_IMAGE_NAMES:
    print(f"Noms d'image dupliques entre dossiers: {len(DUPLICATE_IMAGE_NAMES)}")



ROOT: /Users/mathieu/Documents/CODE/carta_data/cartes_postales_indus/benchmark_300_cartes_vaucluse
Annotation: corpus_benchmark_annotation.json
Corpus dirs: ['corpus_benchmark', 'corpus_benchmark2']
Images indexees: 600
Exports: /Users/mathieu/Documents/CODE/carta_data/cartes_postales_indus/benchmark_300_cartes_vaucluse/exports_benchmark_300


## 1. Charger le fichier d'annotation et preparer la revue visuelle

Cette section sert a verifier que le fichier a bien le bon nombre de lignes et a afficher une carte a la demande.

Pendant l'annotation manuelle, vous pouvez :

- ouvrir `corpus_benchmark_annotation.json` dans l'editeur
- utiliser `review_card(0)` ou `review_card("FRAD084_...")` pour afficher l'image et ses predictions
- revenir ensuite dans le JSON pour remplir les codes d'erreur


In [2]:
records = json.loads(ANNOTATION_PATH.read_text(encoding="utf-8"))
annotation_df = pd.DataFrame(records)
for column in EXPECTED_COLUMNS:
    if column not in annotation_df.columns:
        annotation_df[column] = None
annotation_df = annotation_df[EXPECTED_COLUMNS].copy()

missing_prediction_counts = {
    FIELD_CONFIG[field]["label"]: int(annotation_df[config["pred_col"]].map(is_blank).sum())
    for field, config in FIELD_CONFIG.items()
}
bonus_count = int((annotation_df["bonus"].map(normalize_text) == "find_else").sum())

display(annotation_df.head(5))
display(pd.DataFrame([missing_prediction_counts], index=["predictions_manquantes"]))
display(pd.DataFrame([{"bonus_find_else": bonus_count, "images_indexees": EXPECTED_IMAGE_COUNT}]))

def review_card(identifier, width=900):
    if isinstance(identifier, int):
        row = annotation_df.iloc[identifier]
    else:
        match = annotation_df[annotation_df["File name"] == identifier]
        if match.empty:
            raise KeyError(f"Carte introuvable: {identifier}")
        row = match.iloc[0]
    image_path = IMAGE_PATH_BY_NAME.get(row["File name"])
    display(Markdown(
        f"### {row['File name']}\n"
        f"- Commune predite : `{row['trans_city']}`\n"
        f"- Lieu-dit predit : `{row['trans_hamlet_uniformise']}`\n"
        f"- Monument predit : `{row['trans_monument_uniformise']}`\n"
        f"- error_city : `{row['error_city']}`\n"
        f"- error_hamlet : `{row['error_hamlet']}`\n"
        f"- error_monument : `{row['error_monument']}`\n"
        f"- another_error : `{row['another_error']}`\n"
        f"- bonus : `{row['bonus']}`"
    ))
    if image_path and image_path.exists():
        display(Image(filename=str(image_path), width=width))
    else:
        print(f"Image absente de l'index: {row['File name']}")

print("Exemples: review_card(0)  |  review_card(\"FRAD084_07FI139_0001.jpg\")")


,File name,trans_city,trans_hamlet_uniformise,trans_monument_uniformise,error_city,error_hamlet,error_monument,another_error,bonus
0,FRAD084_07FI002_0001.jpg,Ansouis,Aucun lieu-dit,Château d'Ansouis,None,None,None,None,None
1,FRAD084_07FI002_0002.jpg,Ansouis,Aucun lieu-dit,Château d'Ansouis,None,None,None,None,None
2,FRAD084_07FI002_0003.jpg,Aucune ville,Aucun lieu-dit,Aucun monument,None,None,None,None,None
3,FRAD084_07FI002_0004.jpg,Ansouis,Aucun lieu-dit,Château d'Ansouis,None,None,None,None,None
4,FRAD084_07FI002_0005.jpg,Ansouis,Aucun lieu-dit,Porte fortifiée du Château,None,None,None,None,None


,Commune,Lieu-dit,Monument
predictions_manquantes,0,0,0


,bonus_find_else,images_indexees
0,31,600


Exemples: review_card(0)  |  review_card("FRAD084_07FI139_0001.jpg")


## 2. Annoter manuellement le fichier JSON

A ce stade, completer `corpus_benchmark_annotation.json` en suivant `guide_benchmark.md`.

Regles pratiques :

- laisser `null` si la prediction est acceptable
- renseigner un seul code dans `error_city`, `error_hamlet`, `error_monument`
- utiliser `another_error` pour les erreurs croisees, plusieurs codes possibles separes par `;`
- utiliser `bonus = find_else` quand le modele retrouve correctement un niveau sans texte de localisation exploitable
- pour `FRAD084_07FI007_0147_01.jpg`, la source JSON est absente : `another_error = source_json_absente` est le code recommande si vous gardez cette carte dans le benchmark

Quand l'annotation est terminee, relancer la cellule suivante pour recharger et valider le fichier.


In [3]:
annotation_df = pd.DataFrame(json.loads(ANNOTATION_PATH.read_text(encoding="utf-8")))
for column in EXPECTED_COLUMNS:
    if column not in annotation_df.columns:
        annotation_df[column] = None
annotation_df = annotation_df[EXPECTED_COLUMNS].copy()

validation_rows = []
ambiguous_mask = annotation_df["another_error"].map(lambda value: "cas_ambigu" in parse_multi_codes(value))
ambiguous_count = int(ambiguous_mask.sum())

if len(annotation_df) != EXPECTED_IMAGE_COUNT:
    validation_rows.append({"check": "nb_lignes", "status": "warning", "detail": f"{len(annotation_df)} lignes pour {EXPECTED_IMAGE_COUNT} images indexees"})
else:
    validation_rows.append({"check": "nb_lignes", "status": "ok", "detail": f"{EXPECTED_IMAGE_COUNT} lignes pour {EXPECTED_IMAGE_COUNT} images indexees"})

missing_images = [name for name in annotation_df["File name"] if name not in IMAGE_PATH_BY_NAME]
validation_rows.append({
    "check": "images_associees",
    "status": "ok" if not missing_images else "warning",
    "detail": "toutes les images sont presentes dans l'index" if not missing_images else f"images manquantes: {missing_images[:5]}",
})

validation_rows.append({
    "check": "noms_images_dupliques",
    "status": "ok" if not DUPLICATE_IMAGE_NAMES else "warning",
    "detail": "aucun doublon entre les dossiers corpus_benchmark*" if not DUPLICATE_IMAGE_NAMES else f"doublons: {sorted(DUPLICATE_IMAGE_NAMES)[:5]}",
})

for column in ["error_city", "error_hamlet", "error_monument"]:
    invalid_codes = sorted({
        str(value).strip()
        for value in annotation_df[column]
        if not is_blank(value) and str(value).strip() not in ALLOWED_ERRORS[column]
    })
    validation_rows.append({
        "check": f"codes_{column}",
        "status": "ok" if not invalid_codes else "warning",
        "detail": "codes valides" if not invalid_codes else ", ".join(invalid_codes[:10]),
    })

invalid_other = sorted({
    code
    for value in annotation_df["another_error"]
    for code in parse_multi_codes(value)
    if code not in ALLOWED_ERRORS["another_error"]
})
validation_rows.append({
    "check": "codes_another_error",
    "status": "ok" if not invalid_other else "warning",
    "detail": "codes valides" if not invalid_other else ", ".join(invalid_other[:10]),
})

invalid_bonus = sorted({
    str(value).strip()
    for value in annotation_df["bonus"]
    if not is_blank(value) and str(value).strip() not in ALLOWED_BONUS
})
validation_rows.append({
    "check": "codes_bonus",
    "status": "ok" if not invalid_bonus else "warning",
    "detail": "codes valides" if not invalid_bonus else ", ".join(invalid_bonus[:10]),
})

validation_rows.append({
    "check": "cas_ambigu_exclus",
    "status": "ok",
    "detail": f"{ambiguous_count} cartes seront exclues du benchmark quantitatif",
})

validation_df = pd.DataFrame(validation_rows)
display(validation_df)

missing_source_rows = annotation_df[annotation_df["File name"] == "FRAD084_07FI007_0147_01.jpg"]
if not missing_source_rows.empty:
    display(missing_source_rows)


,check,status,detail
0,nb_lignes,ok,600 lignes pour 600 images indexees
1,images_associees,ok,toutes les images sont presentes dans l'index
2,noms_images_dupliques,ok,aucun doublon entre les dossiers corpus_benchm...
3,codes_error_city,ok,codes valides
4,codes_error_hamlet,ok,codes valides
5,codes_error_monument,ok,codes valides
6,codes_another_error,ok,codes valides
7,codes_bonus,ok,codes valides
8,cas_ambigu_exclus,ok,8 cartes seront exclues du benchmark quantitatif


,File name,trans_city,trans_hamlet_uniformise,trans_monument_uniformise,error_city,error_hamlet,error_monument,another_error,bonus
234,FRAD084_07FI007_0147_01.jpg,Aucune ville,Aucun lieu-dit,Aucun monument,None,None,None,None,None


## 3. Calculer les metriques principales

Le benchmark quantitatif exclut les cartes dont `another_error` contient `cas_ambigu`.

On distingue quatre notions :

- **couverture** : le pipeline a fourni une prediction non vide
- **exactitude sur cartes couvertes** : parmi les cartes avec prediction, part des cartes sans erreur manuelle
- **taux de reussite sur total** : part des cartes benchmarkees a la fois couvertes et correctes
- **bonus `find_else`** : cas positifs ou le modele retrouve correctement un niveau sans texte de localisation exploitable

Cette distinction est utile en presentation, car elle separe la qualite semantique, la couverture operationnelle du pipeline et les cas remarquables a valoriser.


In [4]:
ambiguous_mask = annotation_df["another_error"].map(lambda value: "cas_ambigu" in parse_multi_codes(value))
excluded_ambiguous_df = annotation_df.loc[ambiguous_mask].copy()
benchmark_df = annotation_df.loc[~ambiguous_mask].copy()

benchmark_total = len(benchmark_df)
excluded_ambiguous_count = len(excluded_ambiguous_df)

if benchmark_total == 0:
    raise ValueError("Toutes les cartes sont exclues du benchmark via cas_ambigu.")

def field_status(prediction, error_code):
    if is_blank(prediction):
        return "missing_prediction"
    if is_blank(error_code):
        return "correct"
    return "error"

scope_df = pd.DataFrame([
    {"metric": "Images annotees total", "value": int(len(annotation_df))},
    {"metric": "Images exclues (cas_ambigu)", "value": int(excluded_ambiguous_count)},
    {"metric": "Images retenues dans le benchmark", "value": int(benchmark_total)},
])

summary_rows = []
special_rows = []

for field, config in FIELD_CONFIG.items():
    pred_col = config["pred_col"]
    error_col = config["error_col"]
    status_col = f"status_{field}"
    benchmark_df[status_col] = [field_status(pred, err) for pred, err in zip(benchmark_df[pred_col], benchmark_df[error_col])]
    counts = benchmark_df[status_col].value_counts()
    correct = int(counts.get("correct", 0))
    error = int(counts.get("error", 0))
    missing = int(counts.get("missing_prediction", 0))
    covered = correct + error
    summary_rows.append({
        "field": field,
        "label": config["label"],
        "total": benchmark_total,
        "correct": correct,
        "error": error,
        "missing_prediction": missing,
        "coverage_pct": round(100 * covered / benchmark_total, 1),
        "accuracy_on_covered_pct": round(100 * correct / covered, 1) if covered else np.nan,
        "error_on_covered_pct": round(100 * error / covered, 1) if covered else np.nan,
    })

    special_mask = benchmark_df[pred_col].map(normalize_text) == normalize_text(config["special_value"])
    if int(special_mask.sum()) > 0:
        special_status = benchmark_df.loc[special_mask, status_col].value_counts()
        special_rows.append({
            "field": config["label"],
            "special_value": config["special_value"],
            "predicted_count": int(special_mask.sum()),
            "accepted_count": int(special_status.get("correct", 0)),
            "error_count": int(special_status.get("error", 0)),
            "missing_count": int(special_status.get("missing_prediction", 0)),
        })

summary_df = pd.DataFrame(summary_rows)
special_df = pd.DataFrame(special_rows)

status_cols = [f"status_{field}" for field in FIELD_CONFIG]
benchmark_df["nb_correct"] = (benchmark_df[status_cols] == "correct").sum(axis=1)
benchmark_df["nb_error"] = (benchmark_df[status_cols] == "error").sum(axis=1)
benchmark_df["nb_missing"] = (benchmark_df[status_cols] == "missing_prediction").sum(axis=1)
benchmark_df["fully_correct"] = (benchmark_df["nb_correct"] == 3) & (benchmark_df["nb_missing"] == 0)
benchmark_df["has_cross_error"] = benchmark_df["another_error"].map(lambda value: len(parse_multi_codes(value)) > 0)
benchmark_df["bonus_normalized"] = benchmark_df["bonus"].map(normalize_text)
benchmark_df["has_bonus_find_else"] = benchmark_df["bonus_normalized"] == "find_else"

bonus_count = int(benchmark_df["has_bonus_find_else"].sum())
bonus_df = benchmark_df.loc[
    benchmark_df["has_bonus_find_else"],
    ["File name", "trans_city", "trans_hamlet_uniformise", "trans_monument_uniformise", "bonus"],
].copy()
bonus_summary_df = pd.DataFrame([
    {
        "metric": "Cas bonus find_else",
        "count": bonus_count,
        "pct_total": round(100 * bonus_count / benchmark_total, 1),
    }
])

card_level_df = pd.DataFrame([
    {"metric": "Cartes entierement correctes", "value": int(benchmark_df["fully_correct"].sum())},
    {"metric": "Cartes avec au moins une erreur", "value": int((benchmark_df["nb_error"] > 0).sum())},
    {"metric": "Cartes avec prediction manquante", "value": int((benchmark_df["nb_missing"] > 0).sum())},
    {"metric": "Cartes avec erreur croisee", "value": int(benchmark_df["has_cross_error"].sum())},
    {"metric": "Cas bonus find_else", "value": bonus_count},
])

scope_df.to_csv(EXPORT_DIR / "benchmark_scope.csv", index=False)
summary_df.to_csv(EXPORT_DIR / "benchmark_summary_by_level.csv", index=False)
special_df.to_csv(EXPORT_DIR / "benchmark_special_values.csv", index=False)
card_level_df.to_csv(EXPORT_DIR / "benchmark_card_level.csv", index=False)
benchmark_df.to_csv(EXPORT_DIR / "benchmark_rows_enriched.csv", index=False)
bonus_summary_df.to_csv(EXPORT_DIR / "benchmark_bonus_summary.csv", index=False)
bonus_df.to_csv(EXPORT_DIR / "benchmark_bonus_cases.csv", index=False)
excluded_ambiguous_df.to_csv(EXPORT_DIR / "benchmark_excluded_cas_ambigu.csv", index=False)

display(scope_df)
display(summary_df)
display(card_level_df)
display(bonus_summary_df)
if not special_df.empty:
    display(special_df)
if not bonus_df.empty:
    display(bonus_df.head(10))
if not excluded_ambiguous_df.empty:
    display(excluded_ambiguous_df.head(10))



,metric,value
0,Images annotees total,600
1,Images exclues (cas_ambigu),8
2,Images retenues dans le benchmark,592


,field,label,total,correct,error,missing_prediction,coverage_pct,accuracy_on_covered_pct,error_on_covered_pct
0,city,Commune,592,541,51,0,100.0,91.4,8.6
1,hamlet,Lieu-dit,592,542,50,0,100.0,91.6,8.4
2,monument,Monument,592,529,63,0,100.0,89.4,10.6


,metric,value
0,Cartes entierement correctes,454
1,Cartes avec au moins une erreur,138
2,Cartes avec prediction manquante,0
3,Cartes avec erreur croisee,27
4,Cas bonus find_else,31


,metric,count,pct_total
0,Cas bonus find_else,31,5.2


,field,special_value,predicted_count,accepted_count,error_count,missing_count
0,Commune,Aucune ville,56,45,11,0
1,Lieu-dit,Aucun lieu-dit,499,477,22,0
2,Monument,Aucun monument,195,191,4,0


,File name,trans_city,trans_hamlet_uniformise,trans_monument_uniformise,bonus
46,FRAD084_07FI003_0032.jpg,Apt,Saint-Martian,Eglise de Saint-Martian,find_else
55,FRAD084_07FI003_0044.jpg,Cavaillon,Aucun lieu-dit,Café du Palais,find_else
57,FRAD084_07FI003_0046.jpg,Avignon,Aucun lieu-dit,Tour de l'Horloge,find_else
123,FRAD084_07FI007_0038.jpg,Avignon,Aucun lieu-dit,Aucun monument,find_else
125,FRAD084_07FI007_0040_01.jpg,Avignon,Aucun lieu-dit,Aucun monument,find_else
126,FRAD084_07FI007_0041.jpg,Avignon,Aucun lieu-dit,Château d'Avignon,find_else
128,FRAD084_07FI007_0043.jpg,Avignon,Aucun lieu-dit,Château d'Avignon,find_else
156,FRAD084_07FI007_0069_01.jpg,Avignon,Aucun lieu-dit,Eglise de Avignon,find_else
157,FRAD084_07FI007_0070.jpg,Avignon,Aucun lieu-dit,Eglise de Avignon,find_else
158,FRAD084_07FI007_0071.jpg,Avignon,Aucun lieu-dit,Aucun monument,find_else


,File name,trans_city,trans_hamlet_uniformise,trans_monument_uniformise,error_city,error_hamlet,error_monument,another_error,bonus
85,FRAD084_07FI003_0078.jpg,Apt,Aucun lieu-dit,Fontaine & Place St-Pierre,None,lieudit_absent,monument_mauvais_niveau,cas_ambigu,None
299,FRAD084_7FI_005_0001.jpg,Aucune ville,Lom,Aucun monument,None,lieudit_hallucine,None,cas_ambigu,None
326,FRAD084_07FI139_0015.jpg,Fontaine-de-Vaucluse,Aucun lieu-dit,Tombeau de l'évêque Saint-Veran,None,None,None,cas_ambigu,None
345,FRAD084_07FI139_0034.jpg,Carpentras,Aucun lieu-dit,Pétrarque,mauvaise_commune,None,mauvais_monument,cas_ambigu,None
346,FRAD084_07FI139_0035.jpg,Carpentras,Aucun lieu-dit,Laure,None,None,None,cas_ambigu,None
451,FRAD084_07FI139_0140_01.jpg,Aucune ville,La Fontaine,Aucun monument,None,None,None,cas_ambigu,None
525,FRAD084_07FI152_0040.jpg,Carpentras,Aucun lieu-dit,Mont Ventoux,None,None,None,cas_ambigu,None
547,FRAD084_07FI152_0062.jpg,Carpentras,Vendran,Aucun monument,None,None,None,cas_ambigu,None


## 4. Mesurer un rappel proxy par niveau

Cette section compte, par niveau, les cartes pour lesquelles le pipeline renvoie `Aucune ville`, `Aucun lieu-dit` ou `Aucun monument` sur le corpus benchmarke.

A partir de la :

- `total_moins_aucun` = nombre de cartes benchmarkees moins le nombre de champs `Aucune...` / `Aucun...`
- `nb_absent` = nombre de cartes annotees avec `commune_absente`, `lieudit_absent` ou `monument_absent`
- `rapport_total_moins_aucun_sur_absent` = ratio brut demande
- `rappel_proxy` = `total_moins_aucun / (total_moins_aucun + nb_absent)`

Le notebook calcule aussi une version plus stricte qui retire les predictions vides du numerateur.


In [5]:
ABSENT_ERROR_BY_FIELD = {
    "city": "commune_absente",
    "hamlet": "lieudit_absent",
    "monument": "monument_absent",
}

recall_rows = []

for field, config in FIELD_CONFIG.items():
    pred_col = config["pred_col"]
    error_col = config["error_col"]
    special_norm = normalize_text(config["special_value"])

    pred_norm = benchmark_df[pred_col].map(normalize_text)
    special_count = int((pred_norm == special_norm).sum())
    total_moins_aucun = int(benchmark_total - special_count)
    absent_error_code = ABSENT_ERROR_BY_FIELD[field]
    absent_count = int((benchmark_df[error_col].map(normalize_text) == absent_error_code).sum())

    rapport_vs_absent = round(total_moins_aucun / absent_count, 3) if absent_count else np.nan
    rappel_proxy = round(total_moins_aucun / (total_moins_aucun + absent_count), 3) if (total_moins_aucun + absent_count) else np.nan

    recall_rows.append({
        "field": field,
        "label": config["label"],
        "total_benchmark": int(benchmark_total),
        "nb_aucun": special_count,
        "total_moins_aucun": total_moins_aucun,
        "nb_absent": absent_count,
        "rapport_total_moins_aucun_sur_absent": rapport_vs_absent,
        "rappel_proxy": rappel_proxy,
    })

recall_df = pd.DataFrame(recall_rows)
recall_df.to_csv(EXPORT_DIR / "benchmark_recall_proxy.csv", index=False)
display(recall_df)



,field,label,total_benchmark,nb_aucun,total_moins_aucun,nb_absent,rapport_total_moins_aucun_sur_absent,rappel_proxy
0,city,Commune,592,56,536,11,48.727,0.980
1,hamlet,Lieu-dit,592,499,93,23,4.043,0.802
2,monument,Monument,592,195,397,4,99.250,0.990


## 5. Analyser la typologie d'erreurs

Cette section compte les erreurs par type pour chaque niveau, puis les erreurs croisees declarees dans `another_error`, sur le corpus benchmarke apres exclusion des `cas_ambigu`.


In [6]:
error_type_frames = []

for field, config in FIELD_CONFIG.items():
    error_col = config["error_col"]
    counter = Counter(str(value).strip() for value in benchmark_df[error_col] if not is_blank(value))
    df_errors = pd.DataFrame([
        {"field": config["label"], "error_type": error_type, "count": count}
        for error_type, count in counter.items()
    ])
    if not df_errors.empty:
        df_errors = df_errors.sort_values(["count", "error_type"], ascending=[False, True])
    else:
        df_errors = pd.DataFrame(columns=["field", "error_type", "count"])
    error_type_frames.append(df_errors)
    display(df_errors)

error_types_df = pd.concat(error_type_frames, ignore_index=True) if error_type_frames else pd.DataFrame(columns=["field", "error_type", "count"])
if not error_types_df.empty:
    error_types_pct_df = error_types_df.copy()
    error_types_pct_df["pct_of_benchmark"] = (100 * error_types_pct_df["count"] / benchmark_total).round(2)
else:
    error_types_pct_df = pd.DataFrame(columns=["field", "error_type", "count", "pct_of_benchmark"])

cross_counter = Counter(code for value in benchmark_df["another_error"] for code in parse_multi_codes(value))
cross_df = pd.DataFrame([
    {"error_type": error_type, "count": count}
    for error_type, count in cross_counter.items()
])
if not cross_df.empty:
    cross_df = cross_df.sort_values(["count", "error_type"], ascending=[False, True])
else:
    cross_df = pd.DataFrame(columns=["error_type", "count"])

hallucination_rows = []
for label, error_code in HALLUCINATION_CODES.items():
    count = int(
        error_types_df.loc[
            (error_types_df["field"] == label) & (error_types_df["error_type"] == error_code),
            "count",
        ].sum()
    )
    hallucination_rows.append({
        "field": label,
        "error_type": error_code,
        "count": count,
    })

hallucination_df = pd.DataFrame(hallucination_rows)
hallucination_total = int(hallucination_df["count"].sum())
total_predictions = int(benchmark_total * len(FIELD_CONFIG))
hallucination_summary_df = pd.DataFrame([
    {
        "benchmark_total": int(benchmark_total),
        "nb_niveaux": int(len(FIELD_CONFIG)),
        "total_predictions": total_predictions,
        "hallucination_total": hallucination_total,
        "hallucination_pct_of_total_predictions": round(100 * hallucination_total / total_predictions, 2) if total_predictions else np.nan,
    }
])

special_quality_rows = []
for _, row in special_df.iterrows():
    predicted = int(row["predicted_count"])
    accepted = int(row["accepted_count"])
    special_quality_rows.append({
        "label": row["field"],
        "special_value": row["special_value"],
        "predicted_count": predicted,
        "accepted_count": accepted,
        "special_correct_pct": round(100 * accepted / predicted, 2) if predicted else np.nan,
    })
special_quality_df = pd.DataFrame(special_quality_rows)

quality_metric_rows = []
for _, row in summary_df.iterrows():
    label = row["label"]
    recall_value = recall_df.loc[recall_df["label"] == label, "rappel_proxy"]
    recall_pct = float(recall_value.iloc[0] * 100) if not recall_value.empty and not np.isnan(recall_value.iloc[0]) else np.nan
    halluc_count = int(hallucination_df.loc[hallucination_df["field"] == label, "count"].sum())
    halluc_pct = round(100 * halluc_count / benchmark_total, 2) if benchmark_total else np.nan
    special_pct_series = special_quality_df.loc[special_quality_df["label"] == label, "special_correct_pct"]
    special_pct = float(special_pct_series.iloc[0]) if not special_pct_series.empty else np.nan
    quality_metric_rows.append({
        "label": label,
        "exactitude_sur_couvert_pct": float(row["accuracy_on_covered_pct"]),
        "rappel_proxy_pct": recall_pct,
        "hallucination_pct": halluc_pct,
        "valeurs_aucune_correctes_pct": special_pct,
    })
quality_metrics_df = pd.DataFrame(quality_metric_rows)
quality_metrics_long_df = quality_metrics_df.melt(
    id_vars="label",
    var_name="metric",
    value_name="pct",
)

error_types_df.to_csv(EXPORT_DIR / "benchmark_error_types.csv", index=False)
error_types_pct_df.to_csv(EXPORT_DIR / "benchmark_error_types_pct.csv", index=False)
cross_df.to_csv(EXPORT_DIR / "benchmark_cross_errors.csv", index=False)
hallucination_df.to_csv(EXPORT_DIR / "benchmark_hallucinations_by_level.csv", index=False)
hallucination_summary_df.to_csv(EXPORT_DIR / "benchmark_hallucinations_summary.csv", index=False)
special_quality_df.to_csv(EXPORT_DIR / "benchmark_special_quality.csv", index=False)
quality_metrics_df.to_csv(EXPORT_DIR / "benchmark_quality_metrics_by_level.csv", index=False)

display(cross_df)
display(hallucination_df)
display(hallucination_summary_df)
display(error_types_pct_df)
display(quality_metrics_df)


,field,error_type,count
3,Commune,mauvaise_commune,34
2,Commune,commune_absente,11
0,Commune,commune_hallucinee,3
1,Commune,commune_mauvais_niveau,3


,field,error_type,count
1,Lieu-dit,lieudit_absent,23
0,Lieu-dit,mauvais_lieudit,22
3,Lieu-dit,lieudit_mauvais_niveau,4
2,Lieu-dit,lieudit_hallucine,1


,field,error_type,count
5,Monument,mauvais_monument,24
1,Monument,monument_hallucine,14
2,Monument,monument_mauvais_niveau,11
3,Monument,monument_multiple_incomplet,9
0,Monument,monument_absent,4
4,Monument,monument_partiel,1


,error_type,count
2,swap_commune_monument,11
0,swap_lieudit_monument,11
1,swap_commune_lieudit,5


,field,error_type,count
0,Commune,commune_hallucinee,3
1,Lieu-dit,lieudit_hallucine,1
2,Monument,monument_hallucine,14


,benchmark_total,nb_niveaux,total_predictions,hallucination_total,hallucination_pct_of_total_predictions
0,592,3,1776,18,1.01


## 6. Produire les graphiques vendeurs

Cette section exporte plusieurs PNG dans `plots_benchmark_300/`.

Graphiques generes :

- vue d'ensemble par niveau
- repartition correct / erreur / prediction manquante
- typologie d'erreurs par niveau
- synthese carte par carte
- mise en valeur des cas bonus `find_else`
- rappel proxy par niveau


In [7]:
plot_df = summary_df.copy()
y = np.arange(len(plot_df))
height = 0.32

fig, ax = plt.subplots(figsize=(11, 5.5))
ax.barh(y + height / 2, plot_df["coverage_pct"], height=height, color=COLORS["coverage"], label="Couverture")
ax.barh(y - height / 2, plot_df["accuracy_on_covered_pct"], height=height, color=COLORS["accuracy"], label="Exactitude sur cartes couvertes")
ax.set_yticks(y, plot_df["label"])
ax.set_xlim(0, 105)
ax.set_xlabel("Pourcentage")
ax.set_title(f"Performance du pipeline par niveau sur {benchmark_total} cartes benchmarkees")
ax.legend(loc="lower right")
for values, y_shift in [(plot_df["coverage_pct"], height / 2), (plot_df["accuracy_on_covered_pct"], -height / 2)]:
    for idx, value in enumerate(values):
        ax.text(min(value + 1, 101), idx + y_shift, f"{value:.1f}%", va="center", fontsize=10)
fig.tight_layout()
fig.savefig(PLOTS_DIR / "01_overview_by_level.png", dpi=180, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(10.5, 5.5))
ax.bar(plot_df["label"], plot_df["correct"], color=COLORS["correct"], label="Correct")
ax.bar(plot_df["label"], plot_df["error"], bottom=plot_df["correct"], color=COLORS["error"], label="Erreur")
ax.bar(
    plot_df["label"],
    plot_df["missing_prediction"],
    bottom=plot_df["correct"] + plot_df["error"],
    color=COLORS["missing"],
    label="Prediction manquante",
)
ax.set_ylabel("Nombre de cartes")
ax.set_title(f"Repartition des statuts sur {benchmark_total} cartes benchmarkees")
ax.legend(loc="upper right")
for idx, row in plot_df.iterrows():
    total_height = row["correct"] + row["error"] + row["missing_prediction"]
    ax.text(idx, total_height + 4, f"{int(row['correct'])} correct", ha="center", fontsize=10)
fig.tight_layout()
fig.savefig(PLOTS_DIR / "02_status_split.png", dpi=180, bbox_inches="tight")
plt.show()



/var/folders/48/2kqsx2b57m36f1ysdc4rn_900000gn/T/ipykernel_39310/3743070750.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/folders/48/2kqsx2b57m36f1ysdc4rn_900000gn/T/ipykernel_39310/3743070750.py:38: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [8]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5.5), sharex=False)
for ax, (field, config) in zip(axes, FIELD_CONFIG.items()):
    field_df = error_types_df[error_types_df["field"] == config["label"]].head(8).copy()
    if field_df.empty:
        ax.text(0.5, 0.5, "Aucune erreur codee", ha="center", va="center", fontsize=12)
        ax.set_axis_off()
        continue
    field_df = field_df.sort_values("count")
    ax.barh(field_df["error_type"], field_df["count"], color=COLORS["error"])
    ax.set_title(config["label"])
    ax.set_xlabel("Nombre de cartes")
    for y_pos, value in enumerate(field_df["count"]):
        ax.text(value + 0.1, y_pos, str(value), va="center", fontsize=10)
fig.suptitle("Principaux types d'erreurs")
fig.tight_layout()
fig.savefig(PLOTS_DIR / "03_error_types.png", dpi=180, bbox_inches="tight")
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(16, 5.5), sharex=False)
for ax, (field, config) in zip(axes, FIELD_CONFIG.items()):
    field_df = error_types_pct_df[error_types_pct_df["field"] == config["label"]].head(8).copy()
    if field_df.empty:
        ax.text(0.5, 0.5, "Aucune erreur codee", ha="center", va="center", fontsize=12)
        ax.set_axis_off()
        continue
    field_df = field_df.sort_values("pct_of_benchmark")
    ax.barh(field_df["error_type"], field_df["pct_of_benchmark"], color=COLORS["error"])
    ax.set_title(config["label"])
    ax.set_xlabel("% du benchmark")
    ax.set_xlim(0, max(8, field_df["pct_of_benchmark"].max() * 1.25))
    for y_pos, value in enumerate(field_df["pct_of_benchmark"]):
        ax.text(value + 0.08, y_pos, f"{value:.1f}%", va="center", fontsize=10)
fig.suptitle("Principaux types d'erreurs rapportes au benchmark")
fig.tight_layout()
fig.savefig(PLOTS_DIR / "08_error_types_pct.png", dpi=180, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(11, 5.5))
card_plot = card_level_df.copy()
bar_colors = [COLORS["correct"], COLORS["error"], COLORS["missing"], COLORS["cross"], COLORS["bonus"]]
ax.bar(card_plot["metric"], card_plot["value"], color=bar_colors)
ax.set_ylabel("Nombre de cartes")
ax.set_title("Lecture executive du benchmark")
ax.tick_params(axis="x", rotation=15)
for idx, value in enumerate(card_plot["value"]):
    ax.text(idx, value + 3, str(value), ha="center", fontsize=10)
fig.tight_layout()
fig.savefig(PLOTS_DIR / "04_card_level_summary.png", dpi=180, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(8.5, 4.8))
bonus_plot = pd.DataFrame([
    {"metric": "Cas bonus find_else", "value": int(bonus_count)},
    {"metric": "Autres cartes", "value": int(len(benchmark_df) - bonus_count)},
])
ax.bar(bonus_plot["metric"], bonus_plot["value"], color=[COLORS["bonus"], COLORS["missing"]])
ax.set_ylabel("Nombre de cartes")
ax.set_title("Cas bonus a valoriser")
for idx, value in enumerate(bonus_plot["value"]):
    ax.text(idx, value + 3, str(value), ha="center", fontsize=10)
fig.tight_layout()
fig.savefig(PLOTS_DIR / "05_bonus_cases.png", dpi=180, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(10, 4.8))
recall_plot = recall_df.copy()
ax.bar(recall_plot["label"], recall_plot["rappel_proxy"] * 100, color=COLORS["coverage"])
ax.set_ylim(0, 105)
ax.set_ylabel("Pourcentage")
ax.set_title("Rappel proxy par niveau")
for idx, value in enumerate(recall_plot["rappel_proxy"]):
    if not np.isnan(value):
        ax.text(idx, value * 100 + 2, f"{value * 100:.1f}%", ha="center", fontsize=10)
fig.tight_layout()
fig.savefig(PLOTS_DIR / "06_recall_proxy.png", dpi=180, bbox_inches="tight")
plt.show()

if not cross_df.empty:
    fig, ax = plt.subplots(figsize=(9, 4.8))
    cross_plot = cross_df.sort_values("count")
    ax.barh(cross_plot["error_type"], cross_plot["count"], color=COLORS["cross"])
    ax.set_xlabel("Nombre de cartes")
    ax.set_title("Erreurs croisees et cas structurels")
    for y_pos, value in enumerate(cross_plot["count"]):
        ax.text(value + 0.1, y_pos, str(value), va="center", fontsize=10)
    fig.tight_layout()
    fig.savefig(PLOTS_DIR / "07_cross_errors.png", dpi=180, bbox_inches="tight")
    plt.show()

metric_labels = {
    "exactitude_sur_couvert_pct": "Exactitude",
    "rappel_proxy_pct": "Rappel proxy",
    "hallucination_pct": "Hallucinations",
    "valeurs_aucune_correctes_pct": "Valeurs Aucune correctes",
}
metric_colors = {
    "exactitude_sur_couvert_pct": COLORS["accuracy"],
    "rappel_proxy_pct": COLORS["coverage"],
    "hallucination_pct": COLORS["hallucination"],
    "valeurs_aucune_correctes_pct": COLORS["correct"],
}
metric_order = [
    "exactitude_sur_couvert_pct",
    "rappel_proxy_pct",
    "hallucination_pct",
    "valeurs_aucune_correctes_pct",
]

fig, axes = plt.subplots(1, 3, figsize=(16, 5.8), sharex=True)
for ax, (field, config) in zip(axes, FIELD_CONFIG.items()):
    row = quality_metrics_df[quality_metrics_df["label"] == config["label"]]
    if row.empty:
        ax.text(0.5, 0.5, "Donnees absentes", ha="center", va="center", fontsize=12)
        ax.set_axis_off()
        continue
    row = row.iloc[0]
    metrics_df = pd.DataFrame([
        {
            "metric": metric_labels[key],
            "value": float(row[key]),
            "color": metric_colors[key],
        }
        for key in metric_order
    ])
    metrics_df = metrics_df.iloc[::-1]
    ax.barh(metrics_df["metric"], metrics_df["value"], color=metrics_df["color"])
    ax.set_title(config["label"])
    ax.set_xlim(0, 105)
    ax.set_xlabel("Pourcentage")
    for y_pos, value in enumerate(metrics_df["value"]):
        ax.text(min(value + 1, 101), y_pos, f"{value:.1f}%", va="center", fontsize=10)
fig.suptitle("Qualite par categorie")
fig.tight_layout()
fig.savefig(PLOTS_DIR / "09_quality_metrics_by_category.png", dpi=180, bbox_inches="tight")
plt.show()


/var/folders/48/2kqsx2b57m36f1ysdc4rn_900000gn/T/ipykernel_39310/1789447865.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/folders/48/2kqsx2b57m36f1ysdc4rn_900000gn/T/ipykernel_39310/1789447865.py:30: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/folders/48/2kqsx2b57m36f1ysdc4rn_900000gn/T/ipykernel_39310/1789447865.py:44: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


/var/folders/48/2kqsx2b57m36f1ysdc4rn_900000gn/T/ipykernel_39310/1789447865.py:57: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/folders/48/2kqsx2b57m36f1ysdc4rn_900000gn/T/ipykernel_39310/1789447865.py:69: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7. Generer un pitch summary en deux parties

Cette derniere cellule produit un `benchmark_pitch_summary.md` en deux volets :

1. une partie chiffres, dense et exploitable en interne
2. une partie plus explicative, orientee clients et non-specialistes, avec integration des plots


In [9]:
cards_total = len(annotation_df)
benchmark_total = len(benchmark_df)
fully_correct = int(benchmark_df["fully_correct"].sum())
fully_correct_pct = round(100 * fully_correct / benchmark_total, 1) if benchmark_total else np.nan
error_cards = int((benchmark_df["nb_error"] > 0).sum())
missing_cards = int((benchmark_df["nb_missing"] > 0).sum())
cross_errors = int(benchmark_df["has_cross_error"].sum())
bonus_count = int(benchmark_df["has_bonus_find_else"].sum())
hallucination_total = int(hallucination_df["count"].sum())
total_predictions = int(benchmark_total * len(FIELD_CONFIG))
hallucination_pct = round(100 * hallucination_total / total_predictions, 2) if total_predictions else np.nan
confusion_codes = {"mauvaise_commune", "mauvais_lieudit", "mauvais_monument"}
confusion_df = error_types_df[error_types_df["error_type"].isin(confusion_codes)].copy()
confusion_total = int(confusion_df["count"].sum())
confusion_pct = round(100 * confusion_total / total_predictions, 2) if total_predictions else np.nan

plot_rel = {
    "overview": "../plots_benchmark_300/01_overview_by_level.png",
    "status": "../plots_benchmark_300/02_status_split.png",
    "errors": "../plots_benchmark_300/03_error_types.png",
    "card": "../plots_benchmark_300/04_card_level_summary.png",
    "bonus": "../plots_benchmark_300/05_bonus_cases.png",
    "recall": "../plots_benchmark_300/06_recall_proxy.png",
}

summary_lines = [
    f"# Resume benchmark - {cards_total} cartes annotees",
    "",
    "## Chiffres",
    f"- Corpus annote : {cards_total} cartes",
    f"- Cartes exclues pour cas_ambigu : {excluded_ambiguous_count} ({excluded_ambiguous_count / cards_total * 100:.1f}%)",
    f"- Corpus benchmarke : {benchmark_total} cartes",
    f"- Cartes entierement correctes sur les 3 niveaux : {fully_correct} ({fully_correct_pct:.1f}%)",
    f"- Cartes avec au moins une erreur : {error_cards} ({error_cards / benchmark_total * 100:.1f}%)",
    f"- Cartes avec prediction manquante : {missing_cards} ({missing_cards / benchmark_total * 100:.1f}%)",
    f"- Cas bonus find_else : {bonus_count} ({bonus_count / benchmark_total * 100:.1f}%)",
    f"- Cartes avec erreur croisee ou cas structurel : {cross_errors} ({cross_errors / benchmark_total * 100:.1f}%)",
]

for _, row in summary_df.iterrows():
    summary_lines.append(
        f"- {row['label']} : couverture {row['coverage_pct']:.1f}% | exactitude sur couvert {row['accuracy_on_covered_pct']:.1f}%"
    )

summary_lines.append("")
summary_lines.append("## Volumetrie par niveau")
for _, row in summary_df.iterrows():
    summary_lines.append(
        f"- {row['label']} : correct={int(row['correct'])} | erreurs={int(row['error'])} | predictions manquantes={int(row['missing_prediction'])}"
    )

if not special_df.empty:
    summary_lines.append("")
    summary_lines.append("## Valeurs Aucune")
    for _, row in special_df.iterrows():
        predicted = int(row['predicted_count'])
        accepted = int(row['accepted_count'])
        errors = int(row['error_count'])
        accepted_pct = round(100 * accepted / predicted, 1) if predicted else np.nan
        error_pct = round(100 * errors / predicted, 1) if predicted else np.nan
        summary_lines.append(
            f"- {row['field']} : {row['special_value']} = {predicted} | correct={accepted} ({accepted_pct:.1f}%) | erreurs={errors} ({error_pct:.1f}%)"
        )

summary_lines.append("")
summary_lines.append("## Rappel proxy")
for _, row in recall_df.iterrows():
    if np.isnan(row["rappel_proxy"]):
        continue
    summary_lines.append(
        f"- {row['label']} : {row['rappel_proxy'] * 100:.1f}% (base total_moins_aucun) | nb_aucun={int(row['nb_aucun'])} | nb_absent={int(row['nb_absent'])}"
    )

if not cross_df.empty:
    summary_lines.append("")
    summary_lines.append("## Swaps et erreurs croisees")
    for _, row in cross_df.iterrows():
        summary_lines.append(f"- {row['error_type']} : {int(row['count'])} cartes")

summary_lines.append("")
summary_lines.append("## Confusions")
summary_lines.append(
    f"- Erreurs croisees documentees via another_error : {cross_errors} cartes ({cross_errors / benchmark_total * 100:.1f}%)"
)
summary_lines.append(
    f"- Confusions de lecture de toponymes : {confusion_total} sur {total_predictions} predictions ({confusion_pct:.2f}%)"
)
for code in ["mauvaise_commune", "mauvais_lieudit", "mauvais_monument"]:
    count = int(confusion_df.loc[confusion_df['error_type'] == code, 'count'].sum())
    summary_lines.append(f"- {code} : {count}")
summary_lines.append(
    "- Pour l essentiel, ces confusions correspondent a des lectures de noms d imprimeurs, d editeurs, ou a des formulations de type vue depuis X, panorama depuis Y, etc."
)

summary_lines.append("")
summary_lines.append("## Hallucinations")
summary_lines.append(
    f"- Total hallucinations : {hallucination_total} sur {total_predictions} predictions ({hallucination_pct:.2f}%)"
)
for _, row in hallucination_df.iterrows():
    summary_lines.append(f"- {row['field']} : {int(row['count'])} ({row['error_type']})")

if not error_types_df.empty:
    summary_lines.append("")
    summary_lines.append("## Erreurs par niveau")
    for label in [config["label"] for config in FIELD_CONFIG.values()]:
        field_rows = error_types_df[error_types_df["field"] == label]
        for _, row in field_rows.iterrows():
            summary_lines.append(f"- {label} : {row['error_type']} ({int(row['count'])})")

summary_lines.extend([
    "",
    "## Lecture client",
    "Ce benchmark a ete construit sur un corpus large de 600 cartes postales, avec exclusion des seuls cas juges intrinsequement ambigus. Tous les pourcentages presentes ci dessous sont calcules sur le corpus benchmarke, hors cas_ambigu.",
    "",
    f"Le resultat principal est solide : {fully_correct} cartes sur {benchmark_total} benchmarkees sont justes simultanement sur les trois niveaux de precision. A chaque niveau, la couverture est tres elevee sur le lieu dit et le monument, et reste forte sur la commune. Les {bonus_count} cas bonus find_else montrent en plus que le modele sait retrouver un lieu meme quand la carte ne fournit pas de texte de localisation explicite.",
    "",
    f"Le taux d hallucination reste tres faible pour un systeme generatif : {hallucination_total} hallucinations sur {total_predictions} predictions, soit {hallucination_pct:.2f}%. C est un argument fort en contexte client, car il montre que le pipeline ne remplit pas arbitrairement les champs et conserve un comportement globalement fiable.",
    "",
    f"Les confusions residuelles sont elles aussi lisibles et pilotables : {confusion_total} cas sur {total_predictions} predictions, soit {confusion_pct:.2f}%. Elles proviennent pour l essentiel de formulations editoriales ou descriptives comme les noms d imprimeurs et les mentions du type vue depuis X. Cela ouvre des pistes d amelioration tres concretes par filtrage contextuel et regles de desambiguisation.",
    "",
    "Une part significative des erreurs restantes correspond a des erreurs de granularite ou a des swaps entre niveaux, plutot qu a une localisation totalement hors sujet. Le pipeline est donc souvent proche du bon resultat, avec des marges d amelioration credibles par regles de coherence, post traitement et enrichissement par bases externes.",
    "",
    "### 1. Vue d ensemble",
    "Le graphique ci dessous synthetise la couverture et l exactitude sur cartes couvertes pour les trois niveaux de precision. Il donne une lecture immediate de la robustesse globale du pipeline, sans ajouter une troisieme metrique redondante.",
    "",
    f"![Performance par niveau]({plot_rel['overview']})",
    "",
    "### 2. Lecture executive",
    "Cette vue ramasse le benchmark en quelques chiffres simples : part de cartes entierement correctes, volume d erreurs, poids des cas bonus et place des erreurs croisees. C est le visuel le plus utile pour une slide de synthese ou une page de proposition.",
    "",
    f"![Lecture executive]({plot_rel['card']})",
    "",
    "### 3. Rappel et capacite a ne pas manquer un lieu",
    "Le rappel proxy permet d estimer a quel point le pipeline evite de passer a cote d une information de localisation quand elle est effectivement presente. Il ressort particulierement haut pour la commune et le monument, et reste le plus exigeant sur le lieu dit.",
    "",
    f"![Rappel proxy]({plot_rel['recall']})",
    "",
    "### 4. Cas a forte valeur ajoutee",
    "Les cas bonus find_else mettent en evidence la capacite du modele a mobiliser des indices visuels et contextuels au dela du simple texte imprime. Pour un client, cela montre que la solution ne se limite pas a de l OCR, mais apporte une lecture semantique utile sur des documents patrimoniaux complexes.",
    "",
    f"![Cas bonus]({plot_rel['bonus']})",
    "",
    "### 5. Nature des erreurs restantes",
    "Le dernier visuel montre que les erreurs residuelles se concentrent sur quelques familles identifiables. Elles sont donc pilotables et peuvent etre transformees en feuille de route d amelioration.",
    "",
    f"![Types d erreurs]({plot_rel['errors']})",
])

pitch_summary = "\n".join(summary_lines) + "\n"
(EXPORT_DIR / "benchmark_pitch_summary.md").write_text(pitch_summary, encoding="utf-8")
print(pitch_summary)
print(f"Resume ecrit dans: {EXPORT_DIR / 'benchmark_pitch_summary.md'}")


# Resume benchmark - 600 cartes annotees

## Chiffres
- Corpus annote : 600 cartes
- Cartes exclues pour cas_ambigu : 8 (1.3%)
- Corpus benchmarke : 592 cartes
- Cartes entierement correctes sur les 3 niveaux : 454 (76.7%)
- Cartes avec au moins une erreur : 138 (23.3%)
- Cartes avec prediction manquante : 0 (0.0%)
- Cas bonus find_else : 31 (5.2%)
- Cartes avec erreur croisee ou cas structurel : 27 (4.6%)
- Commune : couverture 100.0% | exactitude sur couvert 91.4%
- Lieu-dit : couverture 100.0% | exactitude sur couvert 91.6%
- Monument : couverture 100.0% | exactitude sur couvert 89.4%

## Volumetrie par niveau
- Commune : correct=541 | erreurs=51 | predictions manquantes=0
- Lieu-dit : correct=542 | erreurs=50 | predictions manquantes=0
- Monument : correct=529 | erreurs=63 | predictions manquantes=0

## Valeurs Aucune
- Commune : Aucune ville = 56 | correct=45 (80.4%) | erreurs=11 (19.6%)
- Lieu-dit : Aucun lieu-dit = 499 | correct=477 (95.6%) | erreurs=22 (4.4%)
- Monument : Aucun

In [ ]:
georef_paths = [
    ROOT / "georef_nominatim" / "vaucluse_georef_nominatim4.csv",
    ROOT / "georef_nominatim" / "vaucluse_georef_nominatim3.csv",
]

SPECIAL_VALUES = {
    "trans_city": "Aucune ville",
    "trans_hamlet_uniformise": "Aucun lieu-dit",
    "trans_monument_uniformise": "Aucun monument",
}
LEVEL_ORDER = ["commune", "hameau", "monument"]

def has_value(series, special_value=None):
    values = series.fillna("").astype(str).str.strip()
    mask = values.ne("")
    if special_value is not None:
        mask &= values.ne(special_value)
    return mask

def best_available_level(row):
    if row["has_scraping_edifice"] or row["has_vlm_monument"]:
        return "monument"
    if row["has_vlm_hamlet"]:
        return "hameau"
    if row["has_scraping_city"] or row["has_vlm_city"]:
        return "commune"
    return "aucun"

def normalize_georef_level(source_georef):
    if source_georef in {"edifice", "monument_uniformise"}:
        return "monument"
    if source_georef == "hameau":
        return "hameau"
    if source_georef == "commune (fallback)":
        return "commune"
    return f"autre:{source_georef}"

def pct(part, total):
    return round(100 * part / total, 1) if total else np.nan

autres_mesures_lines = [
    "# Autres mesures benchmark",
    "",
    "Regle de comparaison :",
    "- niveau georeferencement `monument` = `source_georef` egal a `edifice` ou `monument_uniformise`",
    "- meilleur niveau disponible en amont = `monument` si `Édifice` scraping ou `trans_monument_uniformise` VLM ; sinon `hameau` si `trans_hamlet_uniformise` ; sinon `commune` si `Nom géographique` ou `trans_city`",
    "",
]

for georef_path in georef_paths:
    georef_df = pd.read_csv(georef_path)
    georef_df.columns = [col.lstrip("\ufeff") for col in georef_df.columns]

    georef_df["has_scraping_city"] = has_value(georef_df["Nom géographique"])
    georef_df["has_scraping_edifice"] = has_value(georef_df["Édifice"])
    georef_df["has_vlm_city"] = has_value(georef_df["trans_city"], SPECIAL_VALUES["trans_city"])
    georef_df["has_vlm_hamlet"] = has_value(georef_df["trans_hamlet_uniformise"], SPECIAL_VALUES["trans_hamlet_uniformise"])
    georef_df["has_vlm_monument"] = has_value(georef_df["trans_monument_uniformise"], SPECIAL_VALUES["trans_monument_uniformise"])
    georef_df["best_available_level"] = georef_df.apply(best_available_level, axis=1)
    georef_df["georef_level"] = georef_df["source_georef"].fillna("<vide>").map(normalize_georef_level)

    source_counts = georef_df["source_georef"].fillna("<vide>").value_counts(dropna=False)
    best_counts = georef_df["best_available_level"].value_counts(dropna=False)
    comparison_df = pd.crosstab(georef_df["georef_level"], georef_df["best_available_level"]).reindex(index=LEVEL_ORDER, columns=LEVEL_ORDER, fill_value=0)

    total_rows = len(georef_df)
    exact_best = int(sum(comparison_df.loc[level, level] for level in LEVEL_ORDER))
    lower_than_available = int(comparison_df.loc["commune", "hameau"] + comparison_df.loc["commune", "monument"] + comparison_df.loc["hameau", "monument"])

    commune_fallback_df = georef_df[georef_df["source_georef"] == "commune (fallback)"]
    edifice_df = georef_df[georef_df["source_georef"] == "edifice"]
    monument_uniformise_df = georef_df[georef_df["source_georef"] == "monument_uniformise"]
    hameau_df = georef_df[georef_df["source_georef"] == "hameau"]

    scraping_edifice_and_vlm_monument = int((georef_df["has_scraping_edifice"] & georef_df["has_vlm_monument"]).sum())
    scraping_edifice_only = int((georef_df["has_scraping_edifice"] & ~georef_df["has_vlm_monument"]).sum())
    vlm_monument_only = int((~georef_df["has_scraping_edifice"] & georef_df["has_vlm_monument"]).sum())

    autres_mesures_lines.append(f"## {georef_path.name}")
    autres_mesures_lines.append("### Repartition `source_georef`")
    autres_mesures_lines.append(f"- Lignes retenues : {total_rows}")
    for source_georef, count in source_counts.items():
        autres_mesures_lines.append(f"- {source_georef} : {int(count)}")
    autres_mesures_lines.append("")

    autres_mesures_lines.append("### Informations disponibles en amont")
    autres_mesures_lines.append(f"- Scraping `Nom géographique` renseigne une commune : {int(georef_df['has_scraping_city'].sum())}")
    autres_mesures_lines.append(f"- Scraping `Édifice` renseigne un niveau monument : {int(georef_df['has_scraping_edifice'].sum())}")
    autres_mesures_lines.append(f"- VLM `trans_city` renseigne une commune : {int(georef_df['has_vlm_city'].sum())}")
    autres_mesures_lines.append(f"- VLM `trans_hamlet_uniformise` renseigne un hameau : {int(georef_df['has_vlm_hamlet'].sum())}")
    autres_mesures_lines.append(f"- VLM `trans_monument_uniformise` renseigne un monument : {int(georef_df['has_vlm_monument'].sum())}")
    autres_mesures_lines.append("")

    autres_mesures_lines.append("### Correspondance entre `source_georef` et les signaux amont")
    autres_mesures_lines.append(f"- `edifice` : {len(edifice_df)} cas ; `Édifice` scraping present dans {int(edifice_df['has_scraping_edifice'].sum())} cas ; `trans_monument_uniformise` VLM aussi present dans {int(edifice_df['has_vlm_monument'].sum())} cas")
    autres_mesures_lines.append(f"- `monument_uniformise` : {len(monument_uniformise_df)} cas ; `trans_monument_uniformise` VLM present dans {int(monument_uniformise_df['has_vlm_monument'].sum())} cas ; `Édifice` scraping aussi present dans {int(monument_uniformise_df['has_scraping_edifice'].sum())} cas")
    autres_mesures_lines.append(f"- `hameau` : {len(hameau_df)} cas ; `trans_hamlet_uniformise` VLM present dans {int(hameau_df['has_vlm_hamlet'].sum())} cas")
    autres_mesures_lines.append(f"- `commune (fallback)` : {len(commune_fallback_df)} cas ; `Nom géographique` scraping present dans {int(commune_fallback_df['has_scraping_city'].sum())} cas ; `trans_city` VLM aussi present dans {int(commune_fallback_df['has_vlm_city'].sum())} cas")
    autres_mesures_lines.append("")

    autres_mesures_lines.append("### Meilleur niveau disponible avant georeferencement")
    autres_mesures_lines.append(f"- Niveau `monument` disponible : {int(best_counts.get('monument', 0))}")
    autres_mesures_lines.append(f"- Niveau `monument` disponible via scraping et VLM : {scraping_edifice_and_vlm_monument}")
    autres_mesures_lines.append(f"- Niveau `monument` disponible via scraping seul : {scraping_edifice_only}")
    autres_mesures_lines.append(f"- Niveau `monument` disponible via VLM seul : {vlm_monument_only}")
    autres_mesures_lines.append(f"- Niveau `hameau` comme meilleur niveau disponible : {int(best_counts.get('hameau', 0))}")
    autres_mesures_lines.append(f"- Niveau `commune` seul disponible : {int(best_counts.get('commune', 0))}")
    autres_mesures_lines.append("")

    autres_mesures_lines.append("### Comparaison niveau georeference / meilleur niveau disponible")
    autres_mesures_lines.append(f"- Georeferencement au meilleur niveau disponible : {exact_best} / {total_rows} ({pct(exact_best, total_rows):.1f}%)")
    autres_mesures_lines.append(f"- Georeferencement moins precis que le meilleur niveau disponible : {lower_than_available} / {total_rows} ({pct(lower_than_available, total_rows):.1f}%)")
    autres_mesures_lines.append(f"- `commune (fallback)` alors qu'un niveau `monument` etait disponible : {int(comparison_df.loc['commune', 'monument'])}")
    autres_mesures_lines.append(f"- `commune (fallback)` alors qu'un niveau `hameau` etait disponible : {int(comparison_df.loc['commune', 'hameau'])}")
    autres_mesures_lines.append(f"- `hameau` alors qu'un niveau `monument` etait disponible : {int(comparison_df.loc['hameau', 'monument'])}")
    autres_mesures_lines.append(f"- `commune (fallback)` coherent avec un cas `commune` seul : {int(comparison_df.loc['commune', 'commune'])}")
    autres_mesures_lines.append(f"- `hameau` coherent avec un meilleur niveau `hameau` : {int(comparison_df.loc['hameau', 'hameau'])}")
    autres_mesures_lines.append(f"- Niveau `monument` atteint quand `monument` etait le meilleur niveau disponible : {int(comparison_df.loc['monument', 'monument'])}")
    autres_mesures_lines.append("")

    autres_mesures_lines.append("### Tableau croise")
    autres_mesures_lines.append("| Niveau georeferencement | Commune | Hameau | Monument |")
    autres_mesures_lines.append("| --- | ---: | ---: | ---: |")
    for level in LEVEL_ORDER:
        row = comparison_df.loc[level]
        autres_mesures_lines.append(f"| {level.capitalize()} | {int(row['commune'])} | {int(row['hameau'])} | {int(row['monument'])} |")
    autres_mesures_lines.append("")

autres_mesures_path = ROOT / "autres_mesures_benchmark.md"
autres_mesures_content = "\n".join(autres_mesures_lines).strip() + "\n"
autres_mesures_path.write_text(autres_mesures_content, encoding="utf-8")
print(autres_mesures_content)
print(f"Autres mesures ecrites dans: {autres_mesures_path}")
